# Model performance & interpretability over time

Reads **only** synced files under `results/` in this repo. No GCP credentials or retraining required.

In **Colab**, set `REPO_URL` to your fork (or this repo) and run all cells.

In [ ]:
# Colab: clone the repo (skip if running inside the repo locally)
import os
import sys

REPO_URL = os.environ.get("NOTEBOOK_REPO_URL", "https://github.com/YOUR_ORG/YOUR_REPO.git")
BRANCH = os.environ.get("NOTEBOOK_REPO_BRANCH", "main")

if "google.colab" in sys.modules:
    import subprocess
    subprocess.run(["rm", "-rf", "repo"], check=False)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "repo"],
        check=True,
    )
    os.chdir("repo")
else:
    # Running from checkout: stay at repo root
    if os.path.isdir("results"):
        print("Using current directory as repo root.")
    elif os.path.isdir(os.path.join("..", "results")):
        os.chdir("..")
        print("Using parent as repo root.")
    else:
        print("Place results/ under cwd or set NOTEBOOK_REPO_URL for Colab.")

In [ ]:
from pathlib import Path
import glob

import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path(".")
RESULTS = ROOT / "results"
assert RESULTS.is_dir(), f"Missing {RESULTS} — run GitHub sync workflow first."

hist_path = RESULTS / "history" / "metrics_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    if "timestamp_utc" in hist.columns:
        hist["timestamp_utc"] = pd.to_datetime(hist["timestamp_utc"], utc=True, errors="coerce")
    display(hist.tail(10))
else:
    hist = pd.DataFrame()
    print("No metrics_history.csv yet — train + sync to populate results/history/.")

In [ ]:
# Trend: MAE, MAPE, RMSE, Bias as training data grows (each row = one training run)
if hist.empty:
    print("Skip plots: no history.")
else:
    x = range(len(hist))
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    metrics = [
        ("mae", "MAE"),
        ("mape", "MAPE (%)"),
        ("rmse", "RMSE"),
        ("bias", "Bias (mean pred - actual)"),
    ]
    for ax, (col, title) in zip(axes.ravel(), metrics):
        if col in hist.columns:
            ax.plot(x, hist[col].astype(float), marker="o", markersize=3)
            ax.set_title(title)
            ax.set_xlabel("run index (chronological if file is sorted)")
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Permutation importance: compare a few recent runs (timestamped CSVs)
imp_files = sorted(glob.glob(str(RESULTS / "permutation_importance" / "*-permutation_importance.csv")))
if not imp_files:
    print("No permutation_importance files synced yet.")
else:
    last_three = imp_files[-3:]
    fig, ax = plt.subplots(figsize=(10, 6))
    for fp in last_three:
        dfi = pd.read_csv(fp)
        dfi = dfi.sort_values("importance_mean", ascending=True).tail(12)
        label = Path(fp).stem.replace("-permutation_importance", "")
        ax.barh(dfi["feature"], dfi["importance_mean"], alpha=0.5, label=label)
    ax.set_title("Top features (last few runs)")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# PDP images (latest PNGs in results/pdp)
from IPython.display import Image, display

pdp_files = sorted(glob.glob(str(RESULTS / "pdp" / "*.png")))
if not pdp_files:
    print("No PDP PNGs synced yet.")
else:
    for fp in pdp_files[-6:]:
        display(Image(filename=fp))

## Short interpretation (edit after you inspect plots)

- **Metrics drift**: If MAE/RMSE rise on the latest `eval_date_local`, the latest scrape day may be harder to predict (mix shift, outliers) or the model needs more signal.
- **MAPE** is sensitive to cheap listings; compare with MAE.
- **Bias** consistently positive means systematic over-pricing predictions; negative means under-pricing.
- **Permutation importance** shifts show which features the forest relies on as `listings_master_llm.csv` grows.
- **PDPs** summarize marginal effect of top features on the tuned RandomForest; steep slopes imply stronger sensitivity.

_Fill 2–3 sentences for your midterm describing how behavior changed over the runs you plotted._